# Apartment Maintenance & Resident Complaint Management System
## Exploratory Data Analysis & Visualizations
**Course**: 25ITC31 - Foundations of Data Science  
**Scenario**: 18 (Register No.: 18, 48, 78, 108)  
**Institution**: Velalar College of Engineering and Technology (Autonomous), Erode-12  

---
### Objective:
1. Retrieve relational records from SQLite database (`apartments`, `residents`, `maintenance_payments`, `complaints`).
2. Perform data cleaning, handling missing dates, and feature engineering (turnaround days, overdue balances).
3. Conduct Exploratory Data Analysis (EDA) and visualize key operational metrics using Pandas, Matplotlib, and Seaborn.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 6)

# Connect to SQLite database
conn = sqlite3.connect("../database/apartment_system.db")
df_apartments = pd.read_sql_query("SELECT * FROM apartments", conn)
df_residents = pd.read_sql_query("SELECT * FROM residents", conn)
df_payments = pd.read_sql_query("SELECT * FROM maintenance_payments", conn)
df_complaints = pd.read_sql_query("SELECT * FROM complaints", conn)
conn.close()

print(f"Loaded {len(df_apartments)} apartments, {len(df_residents)} residents, {len(df_payments)} payments, {len(df_complaints)} complaints.")

### 1. Data Cleaning and Preprocessing

In [2]:
# Datetime casting
df_payments["due_date"] = pd.to_datetime(df_payments["due_date"])
df_payments["payment_date"] = pd.to_datetime(df_payments["payment_date"])
df_payments["balance"] = df_payments["amount_due"] - df_payments["amount_paid"]

df_complaints["complaint_date"] = pd.to_datetime(df_complaints["complaint_date"])
df_complaints["resolution_date"] = pd.to_datetime(df_complaints["resolution_date"])

# Merge block number for spatial analysis
df_complaints = df_complaints.merge(df_apartments[["apartment_id", "block_number"]], on="apartment_id", how="left")
df_complaints.head()

### 2. Maintenance Fee Payment Status (Donut Chart)

In [3]:
status_counts = df_payments["payment_status"].value_counts()
colors = ["#2ecc71", "#f39c12", "#e74c3c"]
plt.pie(status_counts.values, labels=status_counts.index, autopct='%1.1f%%', colors=colors,
        wedgeprops=dict(width=0.4, edgecolor='white', linewidth=2), startangle=140)
plt.title("Maintenance Fee Payment Status Distribution (Q1 2026)", fontsize=14, fontweight="bold")
plt.show()

### 3. Resident Complaint Frequency by Facility Category

In [4]:
cat_counts = df_complaints["category"].value_counts().sort_values()
plt.barh(cat_counts.index, cat_counts.values, color="#3498db", edgecolor="#2980b9", height=0.6)
plt.title("Frequency of Resident Complaints by Facility Category", fontsize=14, fontweight="bold")
plt.xlabel("Number of Complaints")
plt.ylabel("Facility Category")
plt.show()

### 4. Monthly Maintenance Demand vs Collection Performance

In [5]:
monthly = df_payments.groupby("billing_month")[["amount_due", "amount_paid"]].sum().reset_index()
x = np.arange(len(monthly))
width = 0.35
plt.bar(x - width/2, monthly["amount_due"], width, label="Demand (Due)", color="#95a5a6")
plt.bar(x + width/2, monthly["amount_paid"], width, label="Collection (Paid)", color="#27ae60")
plt.xticks(x, monthly["billing_month"])
plt.ylabel("Amount (₹)")
plt.title("Monthly Maintenance Collection Performance", fontsize=14, fontweight="bold")
plt.legend()
plt.show()

### 5. Facility Complaint Density Heatmap (Block vs Category)

In [6]:
block_matrix = pd.crosstab(df_complaints["block_number"], df_complaints["category"])
sns.heatmap(block_matrix, annot=True, cmap="YlGnBu", fmt="d", linewidths=1, linecolor="white")
plt.title("Facility Complaint Density Heatmap (Block vs Category)", fontsize=14, fontweight="bold")
plt.xlabel("Facility Category")
plt.ylabel("Apartment Block")
plt.show()